# YOLOv8s Training Notebook for Google Colab

This notebook is designed for a Google Colab T4 GPU runtime.

Pipeline:
- clone the project repo into Colab
- install Ultralytics YOLO
- convert `public/annotations/train.json` and `public/annotations/val.json` to YOLO labels
- train `yolov8s.pt`
- validate the trained model
- export predictions back to the project JSON format
- copy `yolov8s.pt`, `best.pt`, and `last.pt` into `models/`

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/sinh2206/Object_Detection.git"
REPO_DIR = Path("/content/Object_Detection")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")
!git log -1 --oneline

In [ ]:
%pip install -q ultralytics pyyaml matplotlib pillow tqdm

import json
import random
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from PIL import Image, ImageDraw
from ultralytics import YOLO

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
PUBLIC_DIR = REPO_DIR / "public"
MODELS_DIR = REPO_DIR / "models"
ANNOTATIONS_DIR = PUBLIC_DIR / "annotations"

TRAIN_JSON = ANNOTATIONS_DIR / "train.json"
VAL_JSON = ANNOTATIONS_DIR / "val.json"
CLASSES_JSON = PUBLIC_DIR / "classes.json"

TRAIN_IMAGE_DIR = PUBLIC_DIR / "train" / "images"
VAL_IMAGE_DIR = PUBLIC_DIR / "val" / "images"
TRAIN_LABEL_DIR = PUBLIC_DIR / "train" / "labels"
VAL_LABEL_DIR = PUBLIC_DIR / "val" / "labels"
DATA_YAML = PUBLIC_DIR / "dataset_yolov8.yaml"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
RUNS_DIR = REPO_DIR / "runs"
RUN_NAME = "yolov8s_t4"
BASE_WEIGHTS = MODELS_DIR / "yolov8s.pt"
BEST_WEIGHTS = MODELS_DIR / "best.pt"
LAST_WEIGHTS = MODELS_DIR / "last.pt"
VAL_PREDICTIONS_JSON = REPO_DIR / "val_predictions.json"
VAL_METRICS_JSON = REPO_DIR / "val_metrics.json"

TRAIN_ARGS = {
    "data": str(DATA_YAML),
    "epochs": 80,
    "imgsz": 640,
    "batch": -1,
    "patience": 20,
    "workers": 4,
    "device": DEVICE,
    "amp": True,
    "cache": "disk",
    "optimizer": "auto",
    "project": str(RUNS_DIR),
    "name": RUN_NAME,
    "exist_ok": True,
    "seed": 42,
    "close_mosaic": 10,
    "plots": True,
    "verbose": True,
}

print(TRAIN_ARGS)

In [ ]:
def ensure_base_weights(target_path: Path) -> Path:
    if target_path.exists():
        return target_path

    _ = YOLO("yolov8s.pt")
    cache_dir = Path.home() / ".cache" / "ultralytics"
    candidates = [REPO_DIR / "yolov8s.pt", Path("yolov8s.pt")]

    if cache_dir.exists():
        candidates.extend(cache_dir.rglob("yolov8s.pt"))

    for candidate in candidates:
        if candidate.exists():
            shutil.copy2(candidate, target_path)
            return target_path

    raise FileNotFoundError("Could not locate yolov8s.pt after download.")


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def reset_label_dir(label_dir: Path) -> None:
    label_dir.mkdir(parents=True, exist_ok=True)
    for txt_path in label_dir.glob("*.txt"):
        txt_path.unlink()


# The dataset uses 1-based inclusive xyxy boxes, e.g. [1, 1, width, height].
# Convert them to the normalized YOLO cx cy w h format.
def xyxy_1based_to_yolo(box, width: int, height: int):
    x1, y1, x2, y2 = [float(value) for value in box]

    x1 = min(max(x1, 1.0), float(width))
    y1 = min(max(y1, 1.0), float(height))
    x2 = min(max(x2, x1), float(width))
    y2 = min(max(y2, y1), float(height))

    x1_zero = x1 - 1.0
    y1_zero = y1 - 1.0
    box_w = max(x2 - x1_zero, 1.0)
    box_h = max(y2 - y1_zero, 1.0)
    center_x = x1_zero + box_w / 2.0
    center_y = y1_zero + box_h / 2.0

    return (
        center_x / width,
        center_y / height,
        box_w / width,
        box_h / height,
    )


def write_yolo_labels(annotation_path: Path, label_dir: Path, class_to_idx: dict[str, int]):
    data = load_json(annotation_path)
    reset_label_dir(label_dir)

    image_by_id = {item["id"]: item for item in data["images"]}
    annotations_by_image = defaultdict(list)
    for ann in data["annotations"]:
        annotations_by_image[ann["image_id"]].append(ann)

    for image_id, image_info in image_by_id.items():
        width = int(image_info["width"])
        height = int(image_info["height"])
        lines = []

        for ann in annotations_by_image.get(image_id, []):
            class_id = class_to_idx[ann["class"]]
            x, y, w, h = xyxy_1based_to_yolo(ann["bbox"], width, height)
            lines.append(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

        label_path = label_dir / f"{Path(image_id).stem}.txt"
        label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")

    return data


classes = load_json(CLASSES_JSON)
class_to_idx = {class_name: idx for idx, class_name in enumerate(classes)}

train_data = write_yolo_labels(TRAIN_JSON, TRAIN_LABEL_DIR, class_to_idx)
val_data = write_yolo_labels(VAL_JSON, VAL_LABEL_DIR, class_to_idx)

DATA_YAML.write_text(
    yaml.safe_dump(
        {
            "path": str(PUBLIC_DIR),
            "train": "train/images",
            "val": "val/images",
            "names": {idx: name for idx, name in enumerate(classes)},
        },
        sort_keys=False,
        allow_unicode=True,
    ),
    encoding="utf-8",
)

BASE_WEIGHTS = ensure_base_weights(BASE_WEIGHTS)

print(DATA_YAML.read_text(encoding="utf-8"))
print(f"Train images: {len(train_data['images'])}, train boxes: {len(train_data['annotations'])}")
print(f"Val images: {len(val_data['images'])}, val boxes: {len(val_data['annotations'])}")
print(f"Base weights: {BASE_WEIGHTS}")
print(f"Classes: {classes}")

In [ ]:
def show_annotation_samples(split_data: dict, max_images: int = 4, seed: int = 42):
    image_by_id = {item["id"]: item for item in split_data["images"]}
    annotations_by_image = defaultdict(list)
    for ann in split_data["annotations"]:
        annotations_by_image[ann["image_id"]].append(ann)

    populated_image_ids = [image_id for image_id, anns in annotations_by_image.items() if anns]
    random.seed(seed)
    sample_ids = random.sample(populated_image_ids, k=min(max_images, len(populated_image_ids)))

    fig, axes = plt.subplots(1, len(sample_ids), figsize=(5 * len(sample_ids), 5))
    if len(sample_ids) == 1:
        axes = [axes]

    for axis, image_id in zip(axes, sample_ids):
        image_info = image_by_id[image_id]
        image_path = PUBLIC_DIR / image_info["file_name"]
        image = Image.open(image_path).convert("RGB")
        drawer = ImageDraw.Draw(image)

        for ann in annotations_by_image[image_id]:
            x1, y1, x2, y2 = ann["bbox"]
            drawer.rectangle([x1 - 1, y1 - 1, x2 - 1, y2 - 1], outline="red", width=3)
            drawer.text((x1, max(1, y1 - 18)), ann["class"], fill="yellow")

        axis.imshow(image)
        axis.set_title(image_id)
        axis.axis("off")

    plt.tight_layout()


show_annotation_samples(train_data)
print("Train class distribution:")
print(Counter(ann["class"] for ann in train_data["annotations"]))

In [ ]:
model = YOLO(str(BASE_WEIGHTS))
train_results = model.train(**TRAIN_ARGS)

run_dir = RUNS_DIR / RUN_NAME
run_weights_dir = run_dir / "weights"
run_best = run_weights_dir / "best.pt"
run_last = run_weights_dir / "last.pt"

if run_best.exists():
    shutil.copy2(run_best, BEST_WEIGHTS)
if run_last.exists():
    shutil.copy2(run_last, LAST_WEIGHTS)

print(f"Run directory: {run_dir}")
print(f"Best checkpoint copied to: {BEST_WEIGHTS}")
print(f"Last checkpoint copied to: {LAST_WEIGHTS}")

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(data=str(DATA_YAML), split="val", device=DEVICE)

summary = {
    "mAP50-95": float(metrics.box.map),
    "mAP50": float(metrics.box.map50),
    "mAP75": float(metrics.box.map75),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
}
summary

In [ ]:
def yolo_xyxy_to_project_box(box, width: int, height: int):
    x1, y1, x2, y2 = [float(value) for value in box]

    xmin = min(max(x1 + 1.0, 1.0), float(width))
    ymin = min(max(y1 + 1.0, 1.0), float(height))
    xmax = min(max(x2, xmin), float(width))
    ymax = min(max(y2, ymin), float(height))

    return [round(xmin, 4), round(ymin, 4), round(xmax, 4), round(ymax, 4)]


val_image_paths = sorted(VAL_IMAGE_DIR.glob("*"))
project_predictions = []

for result in best_model.predict(
    source=[str(path) for path in val_image_paths],
    imgsz=TRAIN_ARGS["imgsz"],
    conf=0.25,
    iou=0.7,
    device=DEVICE,
    verbose=False,
    stream=True,
):
    image_id = Path(result.path).name
    image_height, image_width = result.orig_shape
    boxes = []

    if result.boxes is not None and len(result.boxes) > 0:
        xyxy_list = result.boxes.xyxy.cpu().tolist()
        conf_list = result.boxes.conf.cpu().tolist()
        cls_list = result.boxes.cls.cpu().tolist()

        for box, confidence, class_id in zip(xyxy_list, conf_list, cls_list):
            boxes.append(
                {
                    "class": classes[int(class_id)],
                    "confidence": round(float(confidence), 6),
                    "bbox": yolo_xyxy_to_project_box(box, image_width, image_height),
                }
            )

    project_predictions.append({"image_id": image_id, "boxes": boxes})

VAL_PREDICTIONS_JSON.write_text(
    json.dumps(project_predictions, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

subprocess.run(
    [
        sys.executable,
        str(PUBLIC_DIR / "tools" / "evaluate_predictions.py"),
        "--ground_truth",
        str(VAL_JSON),
        "--predictions",
        str(VAL_PREDICTIONS_JSON),
        "--output",
        str(VAL_METRICS_JSON),
    ],
    check=True,
)

print(f"Saved predictions: {VAL_PREDICTIONS_JSON}")
print(f"Saved metrics: {VAL_METRICS_JSON}")
print(VAL_METRICS_JSON.read_text(encoding="utf-8"))

In [ ]:
preview_paths = [str(path) for path in val_image_paths[:4]]
preview_results = best_model.predict(source=preview_paths, imgsz=TRAIN_ARGS["imgsz"], conf=0.25, device=DEVICE, verbose=False)

fig, axes = plt.subplots(1, len(preview_results), figsize=(6 * len(preview_results), 6))
if len(preview_results) == 1:
    axes = [axes]

for axis, result in zip(axes, preview_results):
    axis.imshow(result.plot())
    axis.set_title(Path(result.path).name)
    axis.axis("off")

plt.tight_layout()

print("Artifacts:")
print(f"- Base weights: {BASE_WEIGHTS}")
print(f"- Best weights: {BEST_WEIGHTS}")
print(f"- Last weights: {LAST_WEIGHTS}")
print(f"- YOLO data config: {DATA_YAML}")
print(f"- Val predictions: {VAL_PREDICTIONS_JSON}")
print(f"- Val metrics: {VAL_METRICS_JSON}")